# Pipeline Fase 2 — Mesin OCR sebagai Variabel pada Klasifikasi Dokumen Multimodal

Satu notebook yang menjalankan seluruh penelitian dari nol sampai hasil akhir:
setup → data → split → EDA → OCR multi-mesin → fitur teks → training unimodal →
dua aturan fusi → uji hipotesis → ringkasan.

**Rancangan** (lihat `review-1/pipeline-usulan/main.pdf` dan
`vault/00-index/rencana-prompt.md`):

| Komponen | Isi |
|---|---|
| Data | Tobacco3482 saja; 800 latih terstratifikasi (80 di antaranya validasi), 2.682 uji; seed 42/43/44; tanpa RVL-CDIP |
| Variabel bebas 1 | Mesin OCR: QS-OCR (acuan, Tesseract 4 dari TIF), Tesseract (JPG), EasyOCR, PaddleOCR |
| Variabel bebas 2 | Aturan fusi: **CONCAT** (fitur, gaya Audebert) vs **LATE** (rata-rata logit, gaya Kashyap) |
| Dikontrol | MobileNetV2, FastText subword + CNN 1D, hiperparameter, teks mentah tanpa praproses |
| Terikat | OA, macro-F1, F1 per kelas, akurasi tiap unimodal, penurunan akurasi saat satu modalitas dikosongkan |
| Uji | rerata ± simpangan baku 3 seed, McNemar berpasangan dengan koreksi Holm per seed |

Hipotesis kerja: **H1** mesin OCR mengubah akurasi cabang teks secara nyata;
**H3** cabang teks yang lebih kuat membuat fusi lebih memakai teks, dan rata-rata logit
lebih tahan terhadap cabang yang timpang daripada konkatenasi. (H2, BERT, adalah
*future work*.)

**Cara pakai:** atur sel `RUN` di bawah, lalu *Run All*. Semua logika ada di `src/`;
notebook ini hanya memanggilnya. Bagian mahal (OCR, fitur, training) **bisa dilanjutkan**:
hasil yang sudah ada di disk dipakai ulang, jadi kalau kernel mati cukup *Run All* lagi.

Aturan proyek: angka di catatan vault hanya disalin dari keluaran notebook ini.

In [ ]:
# ===== Satu-satunya sel yang perlu diatur =====
import os

RUN = {
    # Bawaan dibaca dari run_all.py (--cepat / --timpa); boleh diganti True/False langsung.
    "quick": os.environ.get("APDM_CEPAT") == "1",      # True: 1 seed, epoch dipangkas
    "overwrite": os.environ.get("APDM_TIMPA") == "1",  # True: latih ulang walau hasil ada
    "run_ocr": True,         # False: pakai teks OCR yang sudah ada di data/interim/ocr
    "ocr_engines": ["tesseract", "easyocr", "paddleocr"],   # mesin yang dijalankan
    "ocr_trial_pages": 20,   # uji kecepatan per mesin sebelum run penuh
    "paddle_models": "mobile",   # "mobile" (PP-OCRv5) | "default" (lebih lambat)
    "run_training": True,    # False: berhenti setelah fitur teks (untuk cek estimasi waktu)
    # EDA
    "flag_percentile": 5,    # ekor distribusi yang ditandai (persen), dari data
    "dup_min_corr": 0.98,    # korelasi thumbnail minimum kandidat duplikat
    "dup_min_jaccard": 0.8,  # kemiripan teks minimum untuk mengonfirmasi duplikat
    "stats_width": 1000,     # lebar seragam (px) saat menghitung statistik citra
    "tsne_perplexity": 30,
}

os.environ["APDM_CEPAT"] = "1" if RUN["quick"] else "0"
os.environ["APDM_TIMPA"] = "1" if RUN["overwrite"] else "0"

## 1. Setup lingkungan

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root proyek:", ROOT, "| python", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    print("CATATAN: proyek diuji pada Python 3.12; versi lain mungkin belum punya wheel.")

%pip install -q -r "{ROOT / 'requirements.txt'}"

In [ ]:
# GPU: kalau ada NVIDIA tapi torch versi CPU, pasang ulang dari indeks CUDA.
from src import setup_env as SE

info = SE.torch_info()
print("torch:", info)
if info and not info["cuda"] and SE.has_nvidia():
    print("GPU NVIDIA ada tapi torch tidak melihat CUDA -> pasang ulang dari indeks cu124")
    %pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124
    info = SE.torch_info()
    print("setelah dipasang ulang:", info, "-> RESTART kernel lalu Run All lagi.")
elif not (info and info["cuda"]):
    print("Tidak ada GPU NVIDIA: training akan sangat lambat. Pertimbangkan RUN['quick']=True.")

In [ ]:
# Kredensial Kaggle (citra Tobacco3482). Token tidak pernah dicetak.
import getpass

if SE.kaggle_ready():
    print("kredensial Kaggle sudah ada")
elif (src := SE.adopt_kaggle_json()) is not None:
    print(f"kaggle.json dari {src} dipasang ke {SE.KAGGLE_JSON}")
else:
    print("kaggle.json tidak ditemukan. Isi dari kaggle.com -> Settings -> API.")
    user = input("Kaggle username: ").strip()
    key = getpass.getpass("Kaggle key (tidak terlihat): ").strip()
    if user and key:
        SE.write_kaggle_token(user, key)
    del key
ok, msg = SE.verify_kaggle()
print("kaggle:", "OK" if ok else "GAGAL", "-", msg)
assert ok, "Kredensial Kaggle belum bisa dipakai."

In [ ]:
# Self-check semua modul src/ (tanpa dataset). Gagal di sini = jangan lanjut.
res = SE.run_self_checks()
for m, err in res.items():
    print(("OK    " if err is None else "GAGAL ") + f"src/{m}" + ("" if err is None else f"  -> {err}"))
assert all(e is None for e in res.values()), "Ada self-check yang gagal."

In [ ]:
import json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import data as D, datasets as DS, eda as EDA, experiment as EX, ocr as OCR
from src import plots as P, splits as S, text_features as TF, train as T
from src.config import CONFIG, deviasi_runtime, set_seed
from src.models import CNN1D, ImageBranch, build_transform
from src.utils import save_fig

set_seed(42)
CLASSES = CONFIG["classes"]
SEEDS = CONFIG["seeds"]
CONDITIONS = CONFIG["ocr_conditions"]
REPORTS = CONFIG["fase2_reports"]
for d in (CONFIG["fase2_data"], CONFIG["fase2_models"], REPORTS, CONFIG["data_interim"]):
    d.mkdir(parents=True, exist_ok=True)
device = T.pick_device()

def show(fig, name):
    save_fig(fig, f"fase2_{name}")
    plt.show()

print(f"device {device} | seed {SEEDS} | epoch {CONFIG['epochs']} | kondisi OCR {CONDITIONS}")
print("deviasi runtime (salin ke catatan eksperimen):")
for d in deviasi_runtime() or ["tidak ada"]:
    print("  -", d)

## 2. Data: unduh, audit, pasangkan

Citra JPG dari Kaggle (`patrickaudriaz/tobacco3482jpg`, ± 3,3 GB, sekali saja) dan teks
QS-OCR-Small (2,5 MB). `prepare_dataset` mencari folder kelas, merekonsiliasi nama kelas
(`ADVE` ↔ `Advertisement`), memasangkan citra–teks per stem, lalu menulis
`data/processed/manifest.csv`. Yatim dan duplikat dilaporkan, tidak disembunyikan.

In [ ]:
audit = D.prepare_dataset()
pairing = pd.DataFrame(audit["pairing"]).T[["n_pasangan", "n_citra_yatim", "n_teks_yatim"]]
pairing.loc["TOTAL"] = pairing.sum()
display(pairing)
for name in ("kandidat_citra", "kandidat_teks"):
    if "warning" in audit[name]:
        print("PERINGATAN", name, ":", audit[name]["warning"])
print("stem ganda lintas kelas -> citra:", len(audit["stem_ganda_citra"]),
      "| teks:", len(audit["stem_ganda_teks"]))

manifest = pd.read_csv(audit["manifest"], dtype={"id": str})
labels = manifest["kelas"].tolist()
y_name = manifest["kelas"].to_numpy()
y = DS.label_indices(labels)
N = len(manifest)
assert N == CONFIG["n_samples_expected"], f"{N} pasangan, harapan {CONFIG['n_samples_expected']}"
print(f"{N} pasangan citra-teks, {manifest['kelas'].nunique()} kelas")

## 3. Split

800 dokumen latih (80 di antaranya validasi untuk *early stopping*), sisanya 2.682 uji,
terstratifikasi, tiga seed. Deviasi dari Audebert (k-fold) dicatat di `src/splits.py`.
Indeks split dibekukan ke disk beserta sidik jari manifest.

In [ ]:
S.write_splits(labels)
splits = {s: S.load_split(s, labels) for s in SEEDS}
sp = splits[SEEDS[0]]
print(f"train {len(sp['train'])} | val {len(sp['val'])} | test {len(sp['test'])}")
split_table = EDA.split_table(y_name, splits, CLASSES)
display(split_table)
split_table.to_csv(REPORTS / "eda_pembagian_data.csv", index=False)

## 4. EDA

### 4.1 Distribusi kelas dan contoh pasangan

In [ ]:
counts = manifest["kelas"].value_counts().reindex(CLASSES)
show(P.bar_counts(counts, "Distribusi kelas Tobacco3482"), "eda_distribusi_kelas")
print(f"rasio kelas terbesar : terkecil = {counts.max() / counts.min():.2f}")

pick = manifest.groupby("kelas").head(1).head(4)
items = [(r.path_citra, Path(r.path_teks).read_text(encoding="utf-8", errors="replace"),
          f"{r.kelas} ({r.n_kata} kata)") for r in pick.itertuples()]
show(P.samples(items), "eda_contoh_pasangan")

### 4.2 Modalitas citra

Resolusi (perkiraan DPI dari lebar halaman US Letter), kecerahan, kontras, rasio tinta,
dan ketajaman (variansi Laplacian; rendah = buram). Hasil di-*cache* karena membaca 3.482
JPG memakan beberapa menit.

In [ ]:
cache_img, cache_thumb = CONFIG["data_interim"] / "eda_citra.csv", CONFIG["data_interim"] / "eda_thumb64.npy"
if cache_img.exists() and cache_thumb.exists() and not RUN["overwrite"]:
    img_stats, thumbs = pd.read_csv(cache_img), np.load(cache_thumb)
else:
    rows, thumbs = EDA.scan_images(manifest["path_citra"].tolist(), RUN["stats_width"])
    img_stats = pd.DataFrame(rows)
    img_stats.to_csv(cache_img, index=False)
    np.save(cache_thumb, thumbs)
img_stats["kelas"] = y_name
display(img_stats.drop(columns="kelas").describe().round(2))
show(P.boxes_by_class(img_stats, ["dpi_perkiraan", "kontras", "rasio_tinta", "ketajaman"],
                      CLASSES, log_cols=("ketajaman",), title="Statistik citra per kelas"),
     "eda_statistik_citra")

### 4.3 Modalitas teks (QS-OCR, acuan)

Panjang teks, bagian kata di luar daftar kata bahasa Inggris (OOV, batas bawah galat OCR),
kode ID arsip, dan kata khas tiap kelas (TF-IDF, dipasang pada bagian latih saja).

In [ ]:
known = OCR.known_words()
qs_texts = [Path(p).read_text(encoding="utf-8", errors="replace") for p in manifest["path_teks"]]
txt_stats = pd.DataFrame(EDA.text_stats(qs_texts, known))
txt_stats["kelas"] = y_name
display(txt_stats.drop(columns="kelas").describe().round(3))
txt_stats["n_kata_plus1"] = txt_stats["n_kata"] + 1
show(P.boxes_by_class(txt_stats, ["n_kata_plus1", "oov"], CLASSES, log_cols=("n_kata_plus1",),
                      title="Statistik teks QS-OCR per kelas"), "eda_statistik_teks")
print(f"teks kosong: {(txt_stats.n_kata_alfabet == 0).mean():.1%} | "
      f"> {CONFIG['max_words']} kata (akan dipotong): {(txt_stats.n_kata > CONFIG['max_words']).mean():.1%}")

fit_idx = np.array(sorted(sp["train"] + sp["val"]))
test_idx = np.array(sp["test"])
terms = pd.Series(EDA.top_terms(qs_texts, y_name, fit_idx, CLASSES), name="kata khas (TF-IDF)")
display(terms.to_frame())
terms.to_csv(REPORTS / "eda_kata_khas_per_kelas.csv")

### 4.4 Kurasi: dokumen bermasalah, duplikat, kebocoran split, jalan pintas kode ID

Dokumen **ditandai, tidak dibuang**: derau OCR adalah objek penelitian. Ambang diambil dari
ekor distribusi data (`flag_percentile`), bukan angka karangan. Duplikat = thumbnail
berkorelasi tinggi **dan** teks OCR mirip. Kebocoran = satu anggota pasangan duplikat di
latih/val dan pasangannya di uji. Jalan pintas = seberapa jauh kode ID arsip saja bisa
menebak kelas (Larson et al.).

In [ ]:
cur_in = txt_stats.drop(columns=["kelas", "n_kata_plus1"]).join(img_stats.drop(columns="kelas"))
curated, cinfo = EDA.curate(cur_in, thumbs, qs_texts, splits, RUN["flag_percentile"],
                            RUN["dup_min_corr"], RUN["dup_min_jaccard"])
print("ambang dari data:", {k: round(v, 3) for k, v in cinfo["ambang"].items()})
print(f"kandidat duplikat dari citra: {cinfo['kandidat_citra']} | terkonfirmasi teks: "
      f"{len(cinfo['pasangan_duplikat'])} | pasangan bocor per seed: {cinfo['pasangan_bocor']}")

flags = [c for c in curated.columns if c.startswith("flag_")]
flag_by_class = (curated[flags].groupby(y_name).mean() * 100).reindex(CLASSES).round(1)
flag_by_class.loc["SEMUA"] = (curated[flags].mean() * 100).round(1)
display(flag_by_class)

text_bad = curated.flag_teks_kosong | curated.flag_teks_pendek | curated.flag_oov_tinggi
image_bad = curated.flag_resolusi_rendah | curated.flag_buram
cross = pd.DataFrame({"teks bermasalah, citra baik": text_bad & ~image_bad,
                      "citra bermasalah, teks baik": image_bad & ~text_bad,
                      "keduanya bermasalah": text_bad & image_bad}).groupby(y_name).sum().reindex(CLASSES)
print("Dokumen yang hanya satu modalitasnya bermasalah (tempat fusi seharusnya menolong):")
display(cross)

curated["prefiks_kode_id"] = [EDA.id_prefix(t) for t in qs_texts]
majority = float((y_name[test_idx] == pd.Series(y_name[fit_idx]).mode()[0]).mean())
shortcut = pd.DataFrame([
    {"fitur": "3 digit awal kode ID di teks OCR",
     **EDA.shortcut_probe(curated["prefiks_kode_id"].to_numpy(), y_name, fit_idx, test_idx)},
    {"fitur": "3 karakter awal ID berkas",
     **EDA.shortcut_probe(manifest["id"].str[:3].to_numpy(), y_name, fit_idx, test_idx)},
]).round(4)
shortcut["kelas_mayoritas"] = round(majority, 4)
display(shortcut)

out = manifest[["id", "kelas", "path_citra", "path_teks"]].copy()
for s, spl in splits.items():
    col = np.empty(N, dtype=object)
    for part in ("train", "val", "test"):
        col[spl[part]] = part
    out[f"split_s{s}"] = col
pd.concat([out, curated], axis=1).to_csv(CONFIG["data_processed"] / "manifest_kurasi.csv", index=False)
flag_by_class.to_csv(REPORTS / "eda_flag_per_kelas.csv")
shortcut.to_csv(REPORTS / "eda_jalan_pintas_kode_id.csv", index=False)

## 5. OCR multi-mesin (variabel bebas 1)

Setiap mesin membaca **citra JPG yang sama** dan menulis satu JSON per halaman (teks
dalam urutan baca + kotak posisi untuk tahap tata letak nanti) ke `data/interim/ocr/<mesin>/`.
Satu proses per mesin, bisa dilanjutkan: halaman yang sudah ada dilewati.

- **Tesseract**: `--oem 1 --psm 3 -l eng`, sama dengan QS-OCR; biner dipasang lewat winget di Windows.
- **EasyOCR**: dipasang dengan *constraints* agar numpy/torch tidak ikut berubah.
- **PaddleOCR**: venv terpisah `.venv-paddle` (pin numpy-nya bentrok dengan torch).

Catatan: Tesseract-JPG ≠ QS-OCR. Versi (5 vs 4-beta) **dan** sumber citra (JPG vs TIF)
sama-sama berbeda, jadi selisih keduanya adalah efek gabungan.

In [ ]:
pins = SE.pin_file(CONFIG["data_interim"] / "constraints_ocr.txt")
if RUN["run_ocr"]:
    %pip install -q pytesseract easyocr -c "{pins}"
    TESS = OCR.find_tesseract() if "tesseract" in RUN["ocr_engines"] else None
    if "tesseract" in RUN["ocr_engines"] and not TESS:
        raise RuntimeError("Tesseract tidak ditemukan. Windows: https://github.com/UB-Mannheim/tesseract/wiki ; "
                           "macOS: brew install tesseract ; Linux: apt install tesseract-ocr. "
                           "Atau hapus 'tesseract' dari RUN['ocr_engines'].")
    PADDLE_PY = OCR.ensure_paddle_venv(ROOT) if "paddleocr" in RUN["ocr_engines"] else None
    GPU = torch.cuda.is_available()
    print("tesseract:", TESS, "| paddle python:", PADDLE_PY, "| EasyOCR GPU:", GPU)

In [ ]:
MANIFEST = CONFIG["data_processed"] / "manifest.csv"
OCR_DIR = CONFIG["ocr_dir"]

def run_engine(engine, limit=None):
    return OCR.run_engine_subprocess(
        engine, MANIFEST, OCR_DIR, ROOT, limit=limit, gpu=GPU, tesseract_cmd=TESS,
        python=str(PADDLE_PY) if engine == "paddleocr" else None,
        paddle_models=RUN["paddle_models"])

if RUN["run_ocr"]:
    for engine in RUN["ocr_engines"]:
        print(f"\n=== {engine}: uji coba {RUN['ocr_trial_pages']} halaman")
        run_engine(engine, RUN["ocr_trial_pages"])
    est = pd.DataFrame({e: {"dtk/halaman": OCR.seconds_per_page(OCR_DIR, e)} for e in RUN["ocr_engines"]}).T
    est["perkiraan penuh (jam)"] = est["dtk/halaman"] * N / 3600
    display(est.round(2))

    t0 = time.perf_counter()
    for engine in RUN["ocr_engines"]:
        print(f"\n=== {engine}: penuh")
        print(run_engine(engine))
    print(f"OCR selesai dalam {(time.perf_counter() - t0) / 3600:.2f} jam")

In [ ]:
# Teks per kondisi, sejajar dengan manifest. Halaman yang hilang = berhenti.
texts = {}
for cond in CONDITIONS:
    texts[cond], missing = OCR.condition_texts(cond, manifest["id"].tolist(),
                                               manifest["path_teks"].tolist(), OCR_DIR)
    assert not missing, (f"{cond}: {len(missing)} halaman belum di-OCR (mis. {missing[:3]}). "
                         "Jalankan bagian OCR sampai tuntas; prosesnya bisa dilanjutkan.")
meta = {e: json.loads((OCR_DIR / e / "_meta.json").read_text()) for e in CONDITIONS[1:]
        if (OCR_DIR / e / "_meta.json").exists()}
(REPORTS / "ocr_meta.json").write_text(json.dumps(meta, indent=1))

i0 = 0
print(f"Dokumen {manifest.id[i0]} ({manifest.kelas[i0]}), 300 karakter pertama per kondisi:")
for cond in CONDITIONS:
    print(f"\n--- {cond} ---\n{texts[cond][i0][:300]}")

### 5.1 EDA OCR: proksi kualitas dan kesepakatan antar-mesin

Tobacco3482 tidak punya transkripsi acuan, jadi CER tidak bisa dihitung. Yang dilaporkan
adalah **proksi**: kata terbaca per halaman, halaman kosong, OOV, kecepatan, dan
kesepakatan antar-mesin (median Jaccard himpunan kata). Sepakat tidak berarti benar.

In [ ]:
qual = pd.DataFrame(OCR.quality_rows(texts, manifest["id"].tolist(), labels, known))
summary_ocr = qual.groupby("ocr", sort=False).agg(
    kata_median=("n_kata", "median"), halaman_kosong_persen=("n_kata", lambda s: 100 * (s == 0).mean()),
    oov_median=("oov", "median"), oov_rata2=("oov", "mean")).reindex(CONDITIONS)
summary_ocr["dtk_per_halaman"] = [OCR.seconds_per_page(OCR_DIR, c) if c != "qs-ocr" else np.nan
                                  for c in CONDITIONS]
display(summary_ocr.round(4))

agree = OCR.agreement(texts)
agree_m = pd.DataFrame([[agree[(a, b)] for b in CONDITIONS] for a in CONDITIONS],
                       index=CONDITIONS, columns=CONDITIONS)
display(agree_m.round(3))
oov_class = qual.pivot_table(index="kelas", columns="ocr", values="oov", aggfunc="median").reindex(CLASSES)[CONDITIONS]
display(oov_class.round(3))

show(P.boxes_by_class(qual, ["oov", "n_kata"], CONDITIONS, by="ocr",
                      title="Proksi kualitas per mesin OCR"), "ocr_proksi_kualitas")
show(P.heatmap(agree_m.values, CONDITIONS, CONDITIONS, "Kesepakatan antar-mesin (median Jaccard)"),
     "ocr_kesepakatan")
show(P.heatmap(oov_class.values, CONDITIONS, CLASSES, "OOV median per kelas", vmax=float(np.nanmax(oov_class.values)),
               cmap="magma"), "ocr_oov_per_kelas")
qual.to_csv(REPORTS / "ocr_proksi_per_dokumen.csv", index=False)
summary_ocr.to_csv(REPORTS / "ocr_ringkasan.csv")
agree_m.to_csv(REPORTS / "ocr_kesepakatan.csv")
oov_class.to_csv(REPORTS / "ocr_oov_per_kelas.csv")

## 6. Fitur teks per kondisi OCR

Tokenisasi spaCy (tanda baca dibuang), lalu vektor FastText **subword** `cc.en.300.bin`
(± 7 GB di disk, ± 15 GB di RAM; diunduh sekali). Satu berkas
`sequences_<kondisi>.npy` berukuran 3482 × 500 × 300 float16 (± 1 GB) per kondisi.
**Teks masuk mentah**: angka, *stopword*, dan tanda baca tidak dibuang (beda dengan
Kanchi/Kashyap), supaya efek mesin OCR tidak tercampur efek praproses.

In [ ]:
import gc

t0 = time.perf_counter()
tokenize = TF.spacy_tokenizer()
feats = TF.build_condition_sequences(texts, tokenize, TF.fasttext_embedder, CONFIG["fase2_data"],
                                     overwrite=RUN["overwrite"])
gc.collect()   # model FastText (± 15 GB) dilepas begitu fungsi selesai
print(f"fitur teks selesai dalam {(time.perf_counter() - t0) / 60:.1f} menit")

tok = pd.DataFrame({c: {"token_median": int(np.median(f["n_token"])),
                        "dokumen_tanpa_token_persen": 100 * (f["n_token"] == 0).mean(),
                        f"di_atas_{CONFIG['max_words']}_persen": 100 * (f["n_token"] > CONFIG["max_words"]).mean(),
                        "ukuran_gb": f["path"].stat().st_size / 1e9} for c, f in feats.items()}).T
display(tok.round(2))
tok.to_csv(REPORTS / "fitur_teks_per_kondisi.csv")

### 6.1 EDA lanjutan: keterpisahan kelas per modalitas

*Linear probe* (regresi logistik) dan t-SNE di atas fitur yang **belum dilatih** pada data
kita: MobileNetV2 ImageNet untuk citra, rata-rata vektor FastText untuk teks tiap kondisi
OCR. Ini memperkirakan seberapa banyak informasi kelas yang dibawa tiap modalitas sebelum
training, dan kelas mana yang lebih dipisahkan teks atau citra.

In [ ]:
cache = CONFIG["data_interim"] / "eda_cnn_mobilenetv2.npy"
if cache.exists() and not RUN["overwrite"]:
    f_img = np.load(cache)
else:
    f_img = EDA.pretrained_image_features(manifest["path_citra"].tolist(),
                                          num_workers=CONFIG["num_workers"])
    np.save(cache, f_img)

f_txt = {}
for cond, f in feats.items():
    seq = np.load(f["path"], mmap_mode="r")
    f_txt[cond] = np.concatenate([EDA.mean_nonzero_rows(seq[i:i + 256]) for i in range(0, N, 256)])
    del seq

probe_rows, f1_rows = [], {}
r, f1_rows["citra (MobileNetV2 pretrained)"] = EDA.probe(f_img, y_name, fit_idx, test_idx, CLASSES)
probe_rows.append({"fitur": "citra (MobileNetV2 pretrained)", **r})
for cond in CONDITIONS:
    r, f1_rows[f"teks {cond}"] = EDA.probe(f_txt[cond], y_name, fit_idx, test_idx, CLASSES)
    probe_rows.append({"fitur": f"teks {cond} (rata-rata FastText)", **r})
probe_table = pd.DataFrame(probe_rows).round(4)
probe_table["kelas_mayoritas"] = round(majority, 4)
display(probe_table)
probe_table.to_csv(REPORTS / "eda_probe_fitur.csv", index=False)

show(P.scatter_2d({"Citra: MobileNetV2 pretrained": EDA.tsne_2d(f_img, 42, RUN["tsne_perplexity"]),
                   "Teks QS-OCR: rata-rata FastText": EDA.tsne_2d(f_txt["qs-ocr"], 42, RUN["tsne_perplexity"])},
                  y_name, CLASSES), "eda_tsne")

comp = pd.DataFrame(f1_rows, index=CLASSES)
diff = comp["teks qs-ocr"] - comp["citra (MobileNetV2 pretrained)"]
show(P.signed_bars(diff, "Kelas yang lebih dipisahkan teks (hijau) atau citra (merah)",
                   "F1 probe teks QS-OCR - F1 probe citra"), "eda_komplementaritas")
comp.round(3).to_csv(REPORTS / "eda_f1_probe_per_kelas.csv")

## 7. Training unimodal

- **IMAGE**: MobileNetV2 (ImageNet, *fine-tune* penuh), 384×384, satu model per seed.
- **TEXT**: CNN 1D (4 lapis, kernel 12, 512 kanal) di atas sekuens FastText, satu model
  per kondisi OCR × seed.

SGD momentum 0,9, lr 0,01, batch 40 (paper), *early stopping* pada val (deviasi kita).
Setiap model menyimpan keluaran untuk **semua** dokumen (fitur, logit, dan keluaran untuk
masukan kosong) ke `data/processed/fase2/outputs/`. Model yang sudah punya berkas itu
dilewati, jadi bagian ini aman dijalankan ulang setelah terputus.

Sel pertama memperkirakan durasi dari 2 epoch percobaan per jenis model.

In [ ]:
transform = build_transform()
paths = manifest["path_citra"].tolist()
blank_img = torch.zeros(1, 3, CONFIG["image_size"], CONFIG["image_size"])
blank_txt = torch.zeros(1, CONFIG["embedding_dim"], CONFIG["max_words"])

def image_ds(idx):
    return DS.ImageDataset(paths, y, idx, transform)

def text_ds_for(cond):
    return lambda idx: DS.SequenceDataset(feats[cond]["path"], y, idx)

jobs = [(EX.IMAGE, s, lambda: ImageBranch(pretrained=True), image_ds, blank_img, "image") for s in SEEDS]
jobs += [(EX.text_name(c), s, CNN1D, text_ds_for(c), blank_txt, "text") for c in CONDITIONS for s in SEEDS]
todo = [j for j in jobs if RUN["overwrite"] or not EX.output_path(j[0], j[1]).exists()]
print(f"{len(jobs)} model unimodal, {len(todo)} belum dilatih")

if todo:
    est = {}
    for kind, build, ds in (("image", lambda: ImageBranch(pretrained=True), image_ds),
                            ("text", CNN1D, text_ds_for(CONDITIONS[0]))):
        if any(j[5] == kind for j in todo):
            loaders = DS.make_loaders(ds, sp)
            est[kind] = T.estimate_duration(build(), loaders, CONFIG["epochs"][kind], device=device)
            DS.close_datasets(*(l.dataset for l in loaders.values()))
    hours = sum(est[j[5]]["perkiraan_jam"] for j in todo)
    print(json.dumps(est, indent=1))
    print(f"\nPerkiraan BATAS ATAS {hours:.1f} jam untuk {len(todo)} model "
          f"(early stopping biasanya memangkasnya).")

In [ ]:
unimodal = []
if RUN["run_training"]:
    t0 = time.perf_counter()
    for name, seed, build, ds, blank, kind in jobs:
        print(f"\n[{name} seed {seed}]")
        meta = EX.train_unimodal(name, seed, splits[seed], build, ds, blank,
                                 epochs=CONFIG["epochs"][kind], n_total=N, device=device)
        out = EX.load_outputs(name, seed)
        te = splits[seed]["test"]
        unimodal.append({**meta, **EX.scores(y[te], out["logit"][te].argmax(1))})
        print(f"  test OA {unimodal[-1]['oa']:.4f} | macro-F1 {unimodal[-1]['macro_f1']:.4f}"
              + (" (dari cache)" if meta["dari_cache"] else ""))
    print(f"\nbagian training selesai dalam {(time.perf_counter() - t0) / 3600:.2f} jam")
    unimodal = pd.DataFrame(unimodal)
    display(unimodal[["nama", "seed", "oa", "macro_f1", "best_epoch", "epochs_run", "durasi_detik", "dari_cache"]])
    unimodal.to_csv(REPORTS / "unimodal_per_seed.csv", index=False)
else:
    print("RUN['run_training'] = False -> berhenti sebelum training.")

## 8. Fusi: dua aturan di atas model unimodal yang sama

Untuk setiap kondisi OCR × seed, dihitung pada set uji yang sama:

| Model | Arti |
|---|---|
| `IMAGE`, `TEXT` | akurasi tiap unimodal |
| `LATE` | argmax rata-rata logit IMAGE dan TEXT (Kashyap), tanpa training |
| `CONCAT` | *head* MLP di atas fitur beku: proyeksi 1280→128 citra ⊕ 128 teks (Audebert); dilatih pada bagian latih, *early stopping* pada val |
| `ORACLE` | benar bila salah satu unimodal benar (batas atas fusi pemilih) |
| `*_tanpa_teks` / `*_tanpa_citra` | uji *missing modality*: modalitas itu diganti keluaran model untuk masukan kosong |

Catatan untuk laporan: CONCAT di sini **bukan** end-to-end seperti Audebert. Kedua cabang
sudah dilatih terpisah lalu dibekukan, supaya perbedaan LATE vs CONCAT murni perbedaan
aturan fusi. *Head* dilatih pada fitur data latih yang juga dipakai melatih unimodal,
jadi ia menilai tiap cabang dari perilaku latihnya (sama seperti pada end-to-end).

In [ ]:
assert RUN["run_training"], "Bagian fusi butuh hasil training (RUN['run_training'] = True)."
t0 = time.perf_counter()
grid, preds = EX.run_grid(y, splits, CONDITIONS, SEEDS)
print(f"fusi selesai dalam {time.perf_counter() - t0:.0f} detik")
summary = EX.summarize(grid)
grid.to_csv(REPORTS / "grid_per_seed.csv", index=False)
summary.to_csv(REPORTS / "grid_ringkasan.csv", index=False)

MAIN = ["IMAGE", "TEXT", "LATE", "CONCAT", "ORACLE"]
fmt = lambda r: f"{r.oa_mean:.4f} ± {r.oa_std:.4f}"
main_table = (summary[summary.model.isin(MAIN)].assign(OA=lambda d: d.apply(fmt, axis=1))
              .pivot(index="ocr", columns="model", values="OA").reindex(CONDITIONS)[MAIN])
print("Tabel utama — OA rerata ± simpangan baku antar-seed")
display(main_table)
f1_table = (summary[summary.model.isin(MAIN)].assign(F1=lambda d: d.apply(lambda r: f"{r.f1_mean:.4f} ± {r.f1_std:.4f}", axis=1))
            .pivot(index="ocr", columns="model", values="F1").reindex(CONDITIONS)[MAIN])
print("macro-F1 rerata ± simpangan baku")
display(f1_table)
main_table.to_csv(REPORTS / "tabel_utama_oa.csv")
f1_table.to_csv(REPORTS / "tabel_utama_f1.csv")
show(P.grouped_bars(summary, MAIN, CONDITIONS, "oa", "OA per kondisi OCR (rerata ± std)"), "grid_oa")
show(P.grouped_bars(summary, MAIN, CONDITIONS, "f1", "macro-F1 per kondisi OCR"), "grid_f1")

In [ ]:
# Pemeriksaan kewarasan (rencana Tahap 11): sel QS-OCR x FastText dan IMAGE memakai
# setelan yang sama dengan Fase 1, jadi harus mendekati angka Fase 1 (dijalankan
# 2026-09-30). Selisih besar pada mode penuh berarti ada yang rusak: berhenti, periksa.
FASE1 = {"TEXT": (0.7266, 0.0059), "IMAGE": (0.8086, 0.0210)}
for model, (ref, ref_std) in FASE1.items():
    r = summary[(summary.ocr == "qs-ocr") & (summary.model == model)].iloc[0]
    gap = r.oa_mean - ref
    ok = abs(gap) <= max(0.02, 2 * ref_std)
    print(f"{model:5s} qs-ocr: {r.oa_mean:.4f} ± {r.oa_std:.4f} | Fase 1 {ref:.4f} ± {ref_std:.4f} "
          f"| selisih {gap:+.4f} -> {'OK' if ok else 'MENYIMPANG'}")
    if not ok and not RUN["quick"]:
        print("  PERIKSA sebelum menafsirkan hasil: split, epoch, sekuens QS-OCR, atau micro-batch berbeda?")
if RUN["quick"]:
    print("(mode cepat: epoch dipangkas, selisih besar wajar)")

## 9. Uji hipotesis

### H1 — Mesin OCR mengubah akurasi cabang teks

McNemar berpasangan antar-kondisi OCR pada set uji yang sama, untuk `TEXT` (inti H1) dan
untuk kedua aturan fusi (pelengkap). Koreksi Holm dilakukan di dalam tiap seed
(6 pasangan per keluarga). H1 didukung bila selisih yang signifikan konsisten di ketiga seed.

In [ ]:
h1 = pd.concat([EX.mcnemar_conditions(preds, m, CONDITIONS, SEEDS) for m in ("TEXT", "LATE", "CONCAT")])
h1.to_csv(REPORTS / "uji_h1_mcnemar_antar_ocr.csv", index=False)
display(h1[h1.model == "TEXT"].round(4))
consistent = (h1.groupby(["model", "a", "b"])
                .agg(selisih_oa_rata2=("oa_a", "mean"), oa_b=("oa_b", "mean"),
                     seed_signifikan=("signifikan", "sum"), n_seed=("seed", "size")))
consistent["selisih_oa_rata2"] -= consistent.pop("oa_b")
print("Ringkasan per pasangan (selisih = a - b):")
display(consistent.round(4))

from sklearn.metrics import f1_score

per_class = {}
for cond in CONDITIONS:
    per_class[cond] = np.mean([f1_score(preds[(cond, s)]["y_true"], preds[(cond, s)]["TEXT"],
                                           labels=range(len(CLASSES)), average=None, zero_division=0)
                               for s in SEEDS], axis=0)
per_class = pd.DataFrame(per_class, index=CLASSES)
print("F1 per kelas cabang TEXT (rerata seed)")
display(per_class.round(3))
per_class.to_csv(REPORTS / "f1_per_kelas_text.csv")
show(P.heatmap(per_class.values, CONDITIONS, CLASSES, "F1 per kelas cabang TEXT", cmap="viridis"),
     "f1_per_kelas_text")

### H3 — Apakah teks yang lebih kuat lebih dipakai, dan aturan mana yang lebih tahan?

`turun_tanpa_teks` = OA fusi − OA fusi dengan teks dikosongkan. H3 didukung bila nilainya
naik seiring OA cabang teks, dan bila LATE kehilangan lebih sedikit daripada CONCAT saat
cabangnya timpang. Juga diuji langsung: McNemar LATE vs CONCAT per kondisi (Holm per seed).

In [ ]:
use = EX.modality_use(grid)
use.to_csv(REPORTS / "uji_h3_pemakaian_modalitas.csv", index=False)
display(use.round(4))
use_mean = use.groupby("ocr", sort=False).mean(numeric_only=True).drop(columns="seed").reindex(CONDITIONS)
print("Rerata antar-seed")
display(use_mean.round(4))
for rule in ("late", "concat"):
    rho = use[["oa_text", f"{rule}_turun_tanpa_teks"]].corr(method="spearman").iloc[0, 1]
    print(f"Spearman(OA teks, {rule.upper()} turun tanpa teks) = {rho:.3f}  (n = {len(use)} sel)")
show(P.modality_use_plot(use), "h3_pemakaian_teks")

h3 = EX.mcnemar_models(preds, "LATE", "CONCAT", CONDITIONS, SEEDS)
h3.to_csv(REPORTS / "uji_h3_mcnemar_late_vs_concat.csv", index=False)
display(h3.round(4))

In [ ]:
# Matriks konfusi sel terbaik (OA LATE/CONCAT tertinggi, rerata seed), seed pertama
best = summary[summary.model.isin(["LATE", "CONCAT"])].sort_values("oa_mean").iloc[-1]
p = preds[(best.ocr, SEEDS[0])]
res = {"y_true": p["y_true"], "y_pred": p[best.model], **EX.scores(p["y_true"], p[best.model])}
show(T.plot_confusion(res, f"{best.model} — OCR {best.ocr} — seed {SEEDS[0]}"), "konfusi_terbaik")
print(T.report(res))

## 10. Ringkasan untuk catatan vault

Salin blok di bawah ke `vault/30-eksperimen/` (field Dataview) dan
`vault/50-hasil/`, lalu ubah status `belum-dijalankan` → `selesai`. Semua CSV dan gambar
ada di `reports/fase2/` dan `reports/figures/fase2_*`.

In [ ]:
print("deviasi runtime:", deviasi_runtime() or "tidak ada")
print(f"seed:: {SEEDS} | epoch:: {CONFIG['epochs']} | split:: 800 latih (80 val) / {len(sp['test'])} uji\n")
for cond in CONDITIONS:
    print(f"--- OCR {cond} ---")
    for m in MAIN:
        r = summary[(summary.ocr == cond) & (summary.model == m)].iloc[0]
        print(f"  model:: {m:7s} oa:: {r.oa_mean:.4f} ± {r.oa_std:.4f}  macro_f1:: {r.f1_mean:.4f} ± {r.f1_std:.4f}")
print("\nH1 (TEXT, pasangan signifikan per seed):")
print(consistent.loc["TEXT"][["selisih_oa_rata2", "seed_signifikan", "n_seed"]].round(4).to_string())
print("\nH3 (rerata turun tanpa teks):")
print(use_mean[["oa_text", "late_turun_tanpa_teks", "concat_turun_tanpa_teks"]].round(4).to_string())
summary_json = {"main_oa": main_table.to_dict(), "main_f1": f1_table.to_dict(),
                "deviasi": deviasi_runtime(), "seeds": SEEDS, "epochs": CONFIG["epochs"]}
(REPORTS / "ringkasan.json").write_text(json.dumps(summary_json, indent=1, ensure_ascii=False))
print("\ndisimpan: reports/fase2/ringkasan.json")